In [1]:
import random 
import json
import pickle
import numpy as np
import nltk
from nltk.stem import WordNetLemmatizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Activation,Dropout
from tensorflow.keras.optimizers import SGD

In [2]:
lemmatizer = WordNetLemmatizer()

In [3]:
intents = json.loads(open('questionAndAns.json').read())

In [4]:
words =[]
classes =[]
documents = []
ignore_letters = ['?','!','.',',']

for intent in intents['intents']:
    for pattern in intent['patterns']:
        word_list = nltk.word_tokenize(pattern)
        words.extend(word_list)
        documents.append((word_list, intent['tag']))        
        if intent['tag'] not in classes:
            classes.append(intent['tag'])

In [5]:
words = [lemmatizer.lemmatize(word) for word in words if word not in ignore_letters]
words = sorted(set(words))

classes = sorted(set(classes))

In [6]:
pickle.dump(words,open('words.pkl','wb'))
pickle.dump(classes,open('classes.pkl','wb'))

In [7]:
training = []
output_empty = [0] * len(classes)

for document in documents:
    bag = []
    word_patterns = document[0]
    word_patterns = [lemmatizer.lemmatize(word.lower()) for word in word_patterns]
    for word in words:
        # This MUST happen for every word in 'words' to keep lengths consistent
         if word in word_patterns:
             bag.append(1)
         else:
             bag.append(0)
    output_row = list(output_empty)
    output_row[classes.index(document[1])] = 1
    training.append([bag,output_row])
    
random.shuffle(training)




In [8]:
train_x = [row[0] for row in training]
train_y = [row[1] for row in training]
train_x = np.array(train_x)
train_y = np.array(train_y)

In [9]:
lengths = [len(bag) for bag in train_x]
print(set(lengths)) 
# If this prints more than one number, your Bag of Words logic has a bug.

{126}


In [10]:
from tensorflow.keras.layers import Input

model = Sequential()
# Modern way to define the input shape
model.add(Input(shape=(len(train_x[0]),))) 
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
# ... rest of your layers
model.add(Dense(64,activation = 'relu'))
model.add(Dropout(0.5))
model.add(Dense(len(train_y[0]),activation='softmax'))

In [11]:
sgd = SGD(learning_rate=0.01, momentum=0.9, nesterov=True)
# Change 'categorical_crossentropyx' to 'categorical_crossentropy'
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])


In [12]:
model.fit(np.array(train_x),np.array(train_y),epochs=75,batch_size=10,verbose=1)
model.save('chatbot_model.keras')
print("DOne ")

Epoch 1/75
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0192 - loss: 3.3542      
Epoch 2/75
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0432 - loss: 3.3114     
Epoch 3/75
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0490 - loss: 3.3101 
Epoch 4/75
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0875 - loss: 3.2857 
Epoch 5/75
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0859 - loss: 3.2737     
Epoch 6/75
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0523 - loss: 3.2370     
Epoch 7/75
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0729 - loss: 3.1953 
Epoch 8/75
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1033 - loss: 3.1617 
Epoch 9/75
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1232 - loss: 3.1755 
Epoch 10/75
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1420 - loss: 3.1476     
Epoch 11/75
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1961 - loss: 3.0677 
Epoch 12/75
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 